# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.


We started with GloVe (2024, Wikipedia + Gigaword, 50 dimensions), the model used in the tutorial, and later also tested fastText (wiki-news, 300 dimensions). Both are trained on large amounts of general English text. ISEAR consists of short personal stories written in everyday English, so a general-purpose English embedding model is a reasonable fit for our data.

To look up a word in the embedding model, we need to split our texts into words in the same way the embedding model’s training data was split. Following the tutorial, we use the basic NLTK tokenizer on lowercased text. We made two versions. For embed-and-pool we remove punctuation and stopwords, because they add little to an average of word vectors. For the LSTM we keep everything, because word order and function words such as “not” can matter there. While checking the unknown words, we noticed that hyphenated words like “girl-friend” were often missing from GloVe, even though both parts were present. We therefore split hyphenated words into their parts before looking them up.

We then measured coverage: how many of the words in our training texts have a vector in the embedding model. Words without a vector are mapped to an unknown token, so their meaning is lost. With the embed-and-pool tokenizer, 99.3% of all tokens in the training texts have a vector (99.7% with the LSTM tokenizer). If we count each distinct word only once, coverage is about 95%. The difference comes from the missing words being rare: almost all of them are spelling mistakes, such as “examn”, “quarelled” and “nieghbour”, that occur only a few times. Since almost every word the model sees has a vector, and the missing ones carry little information, we consider the coverage sufficient.

2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 

To see what the embeddings capture for our task, we compared the GloVe vectors of our seven emotion labels using cosine similarity, and we looked at the nearest neighbours of the matching adjectives. All seven emotions turn out to be quite close to each other, with similarities between 0.53 and 0.87. Even opposite emotions are close: joy and sadness have a similarity of 0.78. This follows from how the embeddings are trained. Words that appear in similar contexts get similar vectors, and “I felt joy” and “I felt sadness” are very similar contexts. The embeddings therefore know that these words are emotions, but they do not separate the emotions well from each other.

The nearest neighbours show the same mixed picture. For “angry” they are good (enraged, furious, outraged), but for “happy” they are mostly common spoken-language words such as “really” and “everyone”. Two results are directly relevant to our task. “Disgusted” and “ashamed” are each other’s neighbours, which matches the confusions we saw in project 1. And the neighbours of “guilty” are pleaded, convicted, charged and murder: GloVe has learned the legal meaning of the word, because that is how it is mostly used in Wikipedia and news. Since every word has only one vector, the emotional meaning we need is mixed with the legal one.

We find this the most troubling limitation for our task: the distinctions we need most, such as shame versus guilt and anger versus disgust, are small in the embedding space, so the classifier gets little help exactly where project 1 already struggled. A smaller limitation is that misspelled words have no vector at all. As task-specific evidence for a better alternative, we compared GloVe 50d with fastText 300d in our classifiers: fastText improved accuracy by about 4 percentage points. Because GloVe 300d could not be downloaded, we cannot tell whether this gain comes from the different model or from the larger dimension.

3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?

In embed-and-pool, each text is represented by the average of its word vectors, and this average is passed to an MLP with two hidden layers. As the tutorial points out, this is still a bag-of-words type encoder: word order is lost, but the word representations are now dense and pretrained instead of counts.

The design choice with by far the largest impact was whether the embeddings are frozen or trainable. With frozen GloVe 50d vectors, embed-and-pool only reached 0.387 accuracy. We think this is because the emotion words lie close together in the GloVe space, and averaging them with the many neutral words in a text washes out the little difference there is. When the embeddings are trainable, the model can move the word vectors so that words linked to different emotions move apart, and accuracy rose to 0.521. Tuning the MLP (dropout 0.5, two hidden layers of 200, 35 epochs) brought this to 0.555. The data size experiment supports this explanation: with frozen embeddings, accuracy stays around 0.40 whether we use 10% or 100% of the training data, while with trainable embeddings it keeps rising with more data.

The second important choice was the embedding model itself. Replacing GloVe 50d by fastText 300d improved both the frozen version (0.555 instead of 0.398) and the trainable version (0.578 instead of 0.551). The trainable fastText model overfits quickly: its best epoch was around epoch 3, after which validation performance went down. Other choices made little difference. Mean pooling was slightly better than sum pooling (0.551 versus 0.544), and a lower learning rate for fastText did not help (0.571 versus 0.578). Both differences are within the variation we see between runs.

Compared to last week, embed-and-pool with GloVe 50d does not improve on our best project 1 system (MLP with token indicators, 0.555). With fastText 300d it does, by about 2 percentage points. So pretrained embeddings only added to our design when they were rich enough and when the model was allowed to adapt them to our task.

4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?

As a recurrent encoder we used an LSTM. Unlike embed-and-pool, the LSTM reads a text word by word and keeps a memory that it updates after every word, so it can use word order. For the LSTM we used the tokenizer that keeps punctuation and stopwords, because words like “not” and “I” can matter once order is taken into account. Unlike embed-and-pool, the LSTM encoder has its own trainable weights, so learning happens not only in the MLP on top but also in how the words are combined.

With the tutorial settings (a small LSTM, a small MLP with little dropout, 20 epochs) the LSTM reached only 0.459 accuracy. We increased the LSTM memory to 100, used a larger MLP with two hidden layers of 200 and dropout 0.5, added word dropout of 0.2 (randomly leaving out words during training) and trained for up to 50 epochs. This raised accuracy to 0.556 with frozen GloVe 50d embeddings. Further changes made little difference: a memory of 200 (0.559), a higher learning rate (0.551) and 100 instead of 50 epochs (0.556) all stayed within the variation we see between runs. Training for 100 epochs gave exactly the same result, because the best epoch was always before epoch 44, so the model had already converged.

Two choices did matter. The first is whether the embeddings are frozen or trainable, and here the LSTM behaves opposite to embed-and-pool: it works best with frozen embeddings. With trainable embeddings, accuracy dropped to 0.520 with GloVe and to 0.509 with fastText, and the best epoch came much earlier (around epoch 10 and 3). This is overfitting: making the embeddings trainable adds hundreds of thousands to millions of parameters, which the model can use to memorise our roughly 4800 training texts per fold. The LSTM can already learn which words matter through its own weights, so it does not need to change the word vectors. The second choice is the embedding model: with frozen fastText 300d vectors, the LSTM reached 0.599, our best result overall.

We also studied the effect of data size by training on 10%, 25%, 50% and 100% of the training part of each fold, while always validating on the full fold. With little data the LSTM performs worse than embed-and-pool, but from about half of the data onwards it catches up and overtakes it. This matches the tutorial’s suggestion that a more complex encoder needs more data to learn how to compose words. At 100% the curves are still rising, so more training data would probably still help.

With GloVe 50d, the LSTM does not improve meaningfully on embed-and-pool (0.556 versus 0.551). With fastText 300d it does, by about 2 percentage points (0.599 versus 0.578). This is why we chose the LSTM with frozen fastText embeddings as our final system. Whether this difference is reliable, and not just variation between runs, is tested with the Bayesian comparison in question 5.

5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?

Our search had two phases. Starting from the tutorial settings, which performed poorly on ISEAR (0.387 for embed-and-pool, 0.459 for the LSTM), we first adjusted several settings at once locally to find a reasonable starting point: trainable embeddings for embed-and-pool, a larger MLP (two layers of 200), dropout 0.5, more epochs, and for the LSTM a memory of 100 and word dropout of 0.2. These became our two base configurations (0.551 and 0.556). In the second phase, run on GPUs, we changed one setting at a time relative to these bases: the embedding model, frozen versus trainable embeddings, LSTM size, learning rate, number of epochs, pooling and the amount of training data. This way, each difference can be attributed to a single choice. Once fastText turned out to give the largest gain, we repeated the most important variations with fastText as the new base. Every configuration was evaluated with 3 seeds × 5 folds, so we always see the variance between runs, and we test whether differences are meaningful with the Bayesian comparison. The search has limitations: in the first phase we changed several settings together, so we cannot tell which of them mattered most, and some settings (MLP size, word dropout, optimizer, weight decay, batch size) were never varied. We also did not compare the LSTM to a simple RNN or a GRU.

To decide which differences are meaningful, we compared each base configuration with its one-change variant using the same correlated Bayesian t-test as in project 1, with a ROPE of 0.005. We consider a difference convincing when its probability is above about 0.95. In the plots, that roughly means the 95% interval lies outside the ROPE.

The large design choices are clearly supported. Replacing GloVe 50d by fastText 300d improves both embed-and-pool (+2.7 points) and the LSTM (+4.3 points) with a probability above 0.99. Freezing the embeddings hurts embed-and-pool, and making them trainable hurts the LSTM, both with probabilities between 0.97 and 1.00. The LSTM is likely better than embed-and-pool when both use fastText (+2.2 points, probability 0.92), but not when both use GloVe, where the comparison is undecided.

For the small tweaks, the comparison cannot reach a decision. For a larger LSTM, a different learning rate, or sum instead of mean pooling, none of the three outcomes (better, practically equal, worse) has a high probability. This does not mean these settings make no difference; it means that 15 measurements per configuration are not enough to tell. The uncertainty about the difference is wider than the ROPE itself, so the test can confirm neither a real effect nor practical equivalence.

Comparing our final system, the LSTM with frozen fastText embeddings and a memory of 100, with every other configuration trained on the full data gives the same picture. It is clearly better than 11 of the 13 alternatives (probability above 0.98), including every GloVe configuration. Only against its two closest competitors is the result less certain: it is likely better than embed-and-pool with fastText (probability 0.92), but against the same LSTM with a memory of 200 the probability is only 0.67, so that comparison remains undecided. We chose the smaller model because it performs at least as well, not because it is clearly better.

Finally, because we selected the best of 32 configurations on cross-validation, its CV score is likely somewhat optimistic. This is consistent with its lower score on the test set (0.583 versus 0.599 in cross-validation).

6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?


For the error analysis we followed the approach of the tutorial. We took the final system, the LSTM with frozen fastText embeddings, and let each of its 15 models (3 seeds × 5 folds) predict the 1504 texts of the test set. For every prediction we recorded whether it was correct, and we described each text with a set of features. From the tutorial we used the number of words (lexicon count), the estimated reading time, two readability scores (Flesch reading ease and Flesch-Kincaid grade) and the label the model predicted. Following the project’s suggestion to look at rare and unknown words, we added two features: the number of words in the text that never occur in the training texts, and the average log frequency of its words in the training texts. We chose these features because they describe how much information a text contains and how unusual its words are, and the predicted label shows which emotions the model handles well. We then fitted the tutorial’s Bayesian logistic regression model, which relates these features to the probability that a prediction is correct. The model converged well: all r_hat values were 1.00 and there were no divergences.

The clearest effect comes from the average frequency of the words in the training texts: texts made up of common words are predicted correctly more often, probably because the model has learned well how to use those words. Texts that contain words never seen in the training texts are also predicted slightly better, even when length and average word frequency are taken into account. This does not mean that an unknown word is better than a known one. Rather, rare words tend to be content words that make the described situation specific, while common words say little about the emotion. The model can use these rare words because its embeddings are pretrained and frozen. Every word in our data has a fastText vector, also when it never occurs in the training texts, and words with a similar meaning have similar vectors. During training, the model learns how vectors in a certain region of this space relate to the emotions, so it can apply this to an unseen word whose vector lies close to words it did see. Because the embeddings are frozen, all vectors stay in the same space, so what the model learned stays valid for words it never saw. Longer texts are predicted correctly more often as well, but this effect is divided between the number of words and the reading time, which measure almost the same thing. > Because our features are strongly related to each other, the individual weights should be interpreted with care: `lexicon_count` and `reading_time` both measure text length, `flesch_reading_ease` and `flesch_kincaid_grade` are both calculated from sentence length and word length, and word length is related to `mean_log_freq`. This is especially true for the two readability scores: `flesch_reading_ease` and `flesch_kincaid_grade` both have negative weights, even though they measure opposite things, so we do not draw conclusions from them. For the predicted label, the intervals are wide, but the direction is consistent: when the model predicts joy, fear or sadness it is more often right, and when it predicts anger, shame or guilt it is more often wrong.

The confusion matrix shows the same pattern. Fear (0.74) and joy (0.71) are recognised best, while shame (0.46) and guilt (0.43) remain the hardest, as in project 1. Many errors are concentrated around anger: 23% of the guilt texts, 18% of the disgust texts and 14% of the shame texts are predicted as anger. Shame and guilt are also still confused with each other. These are all emotions that often describe a conflict or a mistake, so the same kind of situation can lead to different emotions.

Because we have 15 models, we could also check how consistent the errors are. The models mostly agree: 37% of the test texts are predicted correctly by all 15 models, and 20% are predicted wrongly by all 15. Only 14% are uncertain cases, where between 5 and 10 of the models are right. So most errors are not caused by randomness in training, but by texts that are systematically hard for this system. Several of the texts that all models got wrong are ambiguous even for a human reader. For example, “When I was treated improperly by someone” is labelled as fear but predicted as anger, and “Once I saw some children hunting a cat to death” is labelled as anger but predicted as fear. Since the labels in ISEAR are chosen by the writers themselves, some of these errors may reflect the limit of what can be learned from the text alone.

The main limitation of our analysis is that we only analysed the final system, so we cannot yet say which text features explain the difference between the LSTM and embed-and-pool. We prepared the code to repeat the same analysis with the test predictions of embed-and-pool, so that the feature weights of both systems can be compared directly. In the future, we would prioritise three things. First, comparing both architectures on the same features, for example to test whether the LSTM gains most on longer texts, where word order and composition should matter more. Second, adding features that are more specific to our task, such as negation or whether the emotion is named explicitly, which require a part-of-speech tagger or a curated word list. Third, improving the distinction between anger, shame and guilt, for instance with more training data, which our data size experiments suggest would still help, or with contextual embeddings, which can give “guilty” a different meaning in a story than in a courtroom.

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)